# 🌴 OASIS - Ortholog Alignment & Similarity Screener
Automated pipeline for rigorous search and filtering of orthologous sequences using NCBI databases.

In [ ]:
#@title ⚙️ 1. Parameter Configuration

#@markdown Enter the official NCBI ID (RefSeq):
Accession_ID = "NP_001416352.1" #@param {type:"string"}

#@markdown Select the molecule type of your query:
Molecule_Type = "Protein" #@param ["Protein", "Nucleotide"]

#@markdown Set the Minimum Identity and Similarity (%):
Min_Identity = 90 #@param {type:"slider", min:0, max:100, step:1}
Min_Similarity = 95 #@param {type:"slider", min:0, max:100, step:1}

import os
# Exporting variables to the Colab Bash environment
os.environ['ID'] = Accession_ID
os.environ['MOL_TYPE'] = "p" if Molecule_Type == "Protein" else "n"
os.environ['MIN_ID'] = str(Min_Identity)
os.environ['MIN_SIM'] = str(Min_Similarity)
os.environ['FINAL_LIST'] = f"filtered_accessions_ID{Min_Identity}_SIM{Min_Similarity}_{Accession_ID}.txt"

print(f"✅ Parameters saved! ID: {Accession_ID} | Identity: >= {Min_Identity}% | Similarity: >= {Min_Similarity}%")

In [ ]:
%%bash
#@title 📦 2. Tool Installation (Datasets and BLAST+)
echo "Installing NCBI Datasets CLI..."
curl -s -L -o datasets 'https://ftp.ncbi.nlm.nih.gov/pub/datasets/command-line/LATEST/linux-amd64/datasets'
chmod +x datasets

echo "Installing NCBI BLAST+..."
apt-get update -qq && apt-get install -y ncbi-blast+ -qq

echo "✅ Environment successfully configured!"

In [ ]:
%%bash
#@title 📥 3. Sequence and Ortholog Download
export PATH=$PATH:$(pwd)
FASTA_QUERY="query_${ID}.fasta"

echo "🔍 Fetching data from NCBI for ID: $ID..."

if [[ "$MOL_TYPE" == "n" ]]; then
    curl -s "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi?db=nuccore&id=${ID}&rettype=fasta&retmode=text" > "$FASTA_QUERY"
else
    curl -s "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi?db=protein&id=${ID}&rettype=fasta&retmode=text" > "$FASTA_QUERY"
fi

./datasets download gene accession "$ID" --ortholog all --include protein --filename "ortho.zip" > /dev/null 2>&1

if [ ! -s "ortho.zip" ]; then
    echo "❌ Critical Error: Failed to download orthologs. Check if the ID is a valid RefSeq."
    exit 1
fi

unzip -q -o "ortho.zip" -d "ortho_temp"
echo "✅ Target sequence and ortholog family downloaded successfully."

In [ ]:
%%bash
#@title ⚙️ 4. Alignment (BLAST) and Filtering
FASTA_QUERY="query_${ID}.fasta"
ORTHO_FAA=$(find ortho_temp -name "protein.faa" | head -n 1)

if [[ "$MOL_TYPE" == "n" ]]; then BLAST_PROG="blastx"; else BLAST_PROG="blastp"; fi

echo "Running $BLAST_PROG and formatting temporary data..."
makeblastdb -in "$ORTHO_FAA" -dbtype prot -out "temp_db" -parse_seqids -logfile /dev/null

$BLAST_PROG -query "$FASTA_QUERY" -db "temp_db" \
            -outfmt "6 saccver pident ppos" \
            -evalue 1e-5 | \
            awk -v id_min="$MIN_ID" -v sim_min="$MIN_SIM" \
            '($2+0) >= (id_min+0) && ($3+0) >= (sim_min+0) {print $1}' | \
            grep -v "$ID" | sort -u > "$FINAL_LIST"

COUNT=$(wc -l < "$FINAL_LIST")
echo "🎯 Filtering complete! Found $COUNT accessions meeting your strict criteria."

In [ ]:
%%bash
#@title 🧬 5. FASTA File Extraction (Protein and CDS)
export PATH=$PATH:$(pwd)

echo "Extracting Protein FASTA..."
FASTA_FINAL="sequences_PROT_OASIS_${ID}.fasta"
blastdbcmd -db "temp_db" -entry_batch "$FINAL_LIST" -out "$FASTA_FINAL" 2>/dev/null

echo "Downloading Coding Sequences (CDS)..."
CDS_FINAL="sequences_CDS_OASIS_${ID}.fasta"
./datasets download gene accession --inputfile "$FINAL_LIST" --include cds --filename "cds_filtered.zip" > /dev/null 2>&1
unzip -q -o "cds_filtered.zip" -d "cds_temp"
cat $(find cds_temp -name "cds.fna" -o -name "*.fna") > "$CDS_FINAL" 2>/dev/null

# Colab temporary folder cleanup
rm -rf ortho_temp ortho.zip cds_filtered.zip cds_temp temp_db.* "query_${ID}.fasta" datasets

echo "✅ All FASTA files were successfully generated and the environment was cleaned!"

In [ ]:
#@title 💾 6. Download Final Files
from google.colab import files
import os

print("Downloading the generated files:")
arquivos = [
    os.environ['FINAL_LIST'],
    f"sequences_PROT_OASIS_{os.environ['ID']}.fasta",
    f"sequences_CDS_OASIS_{os.environ['ID']}.fasta"
]

for arq in arquivos:
    if os.path.exists(arq):
        print(f"Downloading: {arq}")
        files.download(arq)
    else:
        print(f"⚠️ File {arq} not found. It might have returned 0 results.")